# TP 1 : Exploration Dataset

- Explorer HuggingFace
- Charger un Dataset
- Description du contenu du dataset
- Description statistique du dataset
- Analyse tokens vs lemmes
- Distribution des tokens
- Loi de Zipf

### HuggingFace, un acteur central en Traitement Automatique du Langage (TAL)

Visitez [Hugging Face](https://huggingface.co/) et prenez 5 minutes pour explorer les principaux onglets [`Models`](https://huggingface.co/models) et [`Datasets`](https://huggingface.co/datasets)

### Installation des librairies Python utiles pour le TP

In [ ]:
!pip install tqdm jupyter ipywidgets datasets pyarrow huggingface_hub pandas matplotlib numpy

In [ ]:
from tqdm.auto import tqdm

### Charger un jeu de données à partir de HugginFace

In [ ]:
from datasets import load_dataset_builder

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

builder = load_dataset_builder(dataset_path)

builder = load_dataset_builder(dataset_path,)

print(f"Le jeu de données {dataset_path!r} contient {len(builder.info.splits)} sous-ensembles :")
for name, split in builder.info.splits.items():
    print(f"    {name!r:<20}{split.num_examples:<6,} documents")

Nous commençons par sélectionner l'un des 4 sous-ensembles de 200 documents.

In [ ]:
# sous_ensemble, label_column = "auteur_genre", "auteur_genre"
# sous_ensemble, label_column = "auteur", "auteur"
sous_ensemble, label_column = "date_publication", "date_publication"
# sous_ensemble, label_column = "genre_litteraire", "genre_litteraire"

In [ ]:
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

fichier_local = hf_hub_download(
    repo_id=dataset_path,
    filename=f"data/{sous_ensemble}-00000-of-00001.parquet",
    repo_type="dataset",
    force_download=True,
)

table = pq.read_table(fichier_local)
dataset = table.to_pylist()

print("Nombre de documents :", len(dataset))

def conversions_string_vers_liste(string):
    if isinstance(string, str):
        return string.split(" ")

for document in dataset:
    document["tokens"] = conversions_string_vers_liste(document["tokens"])
    document["lemmes"] = conversions_string_vers_liste(document["lemmes"])

### Structure du dataset

In [ ]:
print(f"Type dataset: {type(dataset)}")
print(f"Longueur dataset: {len(dataset)}")

doc_0 = dataset[0]
print(f"Type 1er document: {type(doc_0)}")
print(f"Clés d'un document: {doc_0.keys()}")

In [ ]:
for key, value in doc_0.items():
    print(f"{key:<20} {type(value)}    {str(value)[:100]}")

In [ ]:
all_publication_date = [doc["date_publication"] for doc in dataset]
print("Min Year\t:", min(all_publication_date))
print("Max Year\t:", max(all_publication_date))
print("Mean Year\t:", int(sum(all_publication_date)/len(all_publication_date)))

In [ ]:
from collections import Counter
all_authors = [doc["auteur"] for doc in dataset]
Counter(all_authors).most_common(10)

In [ ]:
all_authors_gender = [doc["auteur_genre"] for doc in dataset]
Counter(all_authors_gender).most_common()

In [ ]:
all_genre = [doc["genre_litteraire"] for doc in dataset]
Counter(all_genre).most_common()

### Vocabulaire

Construire un dictionnaire du vocabulaire du corpus.

In [ ]:
all_tokens = [token
              for document in dataset
              for token in document["tokens"]
              ]
vocab = sorted(set(all_tokens))
print(f"Taille du corpus: {len(all_tokens):,} tokens")
print(f"Taille du vocabulaire: {len(vocab):,} tokens")

In [ ]:
import pandas as pd
from collections import Counter

vocab_counter = Counter(all_tokens)

df = pd.DataFrame([
    {
        "Rang": i + 1,
        "Token": word,
        "Occurrences": count,
        "Fréquence (%)": 100 * count / len(all_tokens)
    }
    for i, (word, count) in enumerate(vocab_counter)
])

In [ ]:
N = 20
print(df.head(N).to_string(
    index=False,
    formatters={"Fréquence (%)": "{:.2f}%".format},
    justify="center"
))

In [ ]:
target_words = ["homme", "femme", "roi", "reine", "chien", "chat", "cheval", "heureux", "malheureux"]
df[df["Token"].isin(target_words)]

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Global font settings
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
})

frequencies = sorted(vocab_counter.values(), reverse=True)

N = 50000
frequencies = [n * 100 / len(all_tokens) for n in frequencies[:N]]
ranks = np.arange(1, len(frequencies) + 1)

plt.figure(figsize=(10, 5))
plt.bar(ranks, frequencies, width=1.0)

plt.xlabel("Rang")
plt.ylabel("Fréquence")
plt.title(f"Distribution de fréquence : {N} mots les plus fréquents")

plt.xlim(1, N)
plt.grid(axis="y", alpha=0.3)
plt.yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Vocabulaire : {len(vocab_counter):,}")
print(f"Mots apparaissant une seule fois : {sum(c == 1 for c in vocab_counter.values()):,}")
print(f"Mots apparaissant moins de 5 fois : {sum(c < 5 for c in vocab_counter.values()):,}")
print(f"Mots apparaissant moins de 20 fois : {sum(c < 20 for c in vocab_counter.values()):,}")
print(f"Mots apparaissant moins de 100 fois : {sum(c < 100 for c in vocab_counter.values()):,}")
print(f"Mots apparaissant moins de 1 000 fois : {sum(c < 1000 for c in vocab_counter.values()):,}")

### Loi de Zipf

La [loi de Zipf](https://fr.wikipedia.org/wiki/Loi_de_Zipf) est une observation empirique selon laquelle, dans de nombreux ensembles de données classés par ordre décroissant, la fréquence ou la taille d'un élément de rang n est inversement proportionnelle à n. Elle a été formulée en 1932 par le linguiste George Kingsley Zipf (1902-1950) à propos de la fréquence des mots dans un texte.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Global font settings
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 20,
    "axes.labelsize": 17,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

vocab_counter = Counter(all_tokens)
frequencies = np.array(sorted(vocab_counter.values(), reverse=True))
ranks = np.arange(1, len(frequencies) + 1)

# Log-transform
log_ranks = np.log(ranks)
log_freqs = np.log(frequencies)

# Fit a linear regression in log-log space
slope, intercept = np.polyfit(log_ranks, log_freqs, 1)

# Predicted values
fitted_freqs = np.exp(intercept) * ranks ** slope

# Plot
plt.figure(figsize=(10, 5))

plt.loglog(ranks, frequencies, label="Fréquences observées")
plt.loglog(
    ranks,
    fitted_freqs,
    "--",
    linewidth=2,
    label=f"Ajustement en loi de puissance (s={-slope:.3f})"
)

plt.xlabel("Rang")
plt.ylabel("Fréquence")
plt.title("Loi de Zipf dans le corpus")

plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Exposant de Zipf estimé : {-slope:.4f}")